In [6]:
"""
Price Spread Models 40 and 41 — Constant Coefficient
======================================================
Real-time direct h-step forecasts of real TTF NG prices using the
log nominal oil-gas price spread.
Model 40: alpha and beta estimated freely.
Model 41: alpha fixed at 0, beta estimated only.
Expected inflation subtracted via rolling 120-month HICP mean (1-month lag).
Separate OLS per horizon. Expanding window from Feb 2006.
"""

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ── Parameters ────────────────────────────────────────────────────────────────
HORIZONS       = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START     = "2015-01-01"
TRAIN_START    = "2006-02-01"   # first date where pi_bar has 120 obs
PI_BAR_WINDOW  = 120            # rolling window for expected inflation
PRICES_FILE    = "Input_Monthly_Average_Oil_NG_nominal_prices.xlsx"
INFLATION_FILE = "Input_Inflation.xlsx"
REAL_PRICES_FILE = "Input_Real_Average_Monthly_TTF_NG_prices.xlsx"
OUTPUT_FILE    = "Output_PriceSpread_forecasts.xlsx"

# ── Load data ─────────────────────────────────────────────────────────────────
# Nominal prices (for spread)
prices = pd.read_excel(PRICES_FILE, sheet_name="Sheet1")
prices.columns = ["date", "oil_nom", "ng_nom"]
prices["date"]  = pd.to_datetime(prices["date"])
prices = prices.sort_values("date").reset_index(drop=True)

# Log nominal prices and spread
prices["s_ng"]   = np.log(prices["ng_nom"])
prices["s_oil"]  = np.log(prices["oil_nom"])
prices["spread"] = prices["s_ng"] - prices["s_oil"]

# Inflation data
inf = pd.read_excel(INFLATION_FILE, sheet_name="Sheet1")
inf.columns = ["date", "hicp", "inf_pct_change"]
inf["date"]  = pd.to_datetime(inf["date"])
inf = inf.sort_values("date").reset_index(drop=True)

# Rolling pi_bar: 120-month mean of monthly % changes
# At origin t: use pi_bar ending the month before origin (1-month HICP lag)
inf["pi_bar"] = inf["inf_pct_change"].rolling(PI_BAR_WINDOW).mean()

# Map pi_bar to each price observation:
# origin date (end of month M) -> use pi_bar from start of month M - 1
prices["prev_month"] = (prices["date"].dt.to_period("M").dt.to_timestamp()
                        - pd.offsets.MonthBegin(1))
pi_bar_map = inf.set_index("date")["pi_bar"]
prices["pi_bar"] = prices["prev_month"].map(pi_bar_map)

# HICP nowcast for each origin:
# HICP_t_nowcast = HICP_{t-1} * (1 + pi_bar_t)
hicp_map = inf.set_index("date")["hicp"]
prices["hicp_prev"]    = prices["prev_month"].map(hicp_map)
prices["hicp_nowcast"] = prices["hicp_prev"] * (1 + prices["pi_bar"])

# Real TTF price at each origin: R_t = ng_nom_t / hicp_nowcast_t * 100
# (HICP is index 2015=100, so divide by hicp/100)
prices["real_ng"] = prices["ng_nom"] / (prices["hicp_nowcast"] / 100)

# Real actual prices (for evaluation — use final vintage)
real_df = pd.read_excel(REAL_PRICES_FILE, parse_dates=["date"])
real_df = real_df[["date","price_real"]].sort_values("date").reset_index(drop=True)
real_df["date"] = real_df["date"] + pd.offsets.MonthEnd(0)

def get_actual(ym_str):
    m = real_df[real_df["date"].dt.to_period("M").astype(str) == ym_str]
    return m["price_real"].values[0] if len(m) == 1 else np.nan

# ── Training sample ───────────────────────────────────────────────────────────
prices = prices[prices["date"] >= TRAIN_START].dropna(
    subset=["spread","pi_bar","real_ng"]).reset_index(drop=True)

# ── Main forecasting loop ─────────────────────────────────────────────────────
records      = []
eval_origins = prices[prices["date"] >= EVAL_START]["date"].tolist()

print(f"Price Spread forecasting: {len(eval_origins)} origins x "
      f"2 models x {len(HORIZONS)} horizons")
print(f"Models:  Model 40 (alpha+beta), Model 41 (alpha=0, beta only)")
print(f"Direct h-step regression — separate OLS per horizon")
print(f"Training: {TRAIN_START} onwards  |  pi_bar window: {PI_BAR_WINDOW} months")
print()

for i, origin_date in enumerate(eval_origins):

    if i % 20 == 0:
        print(f"  Origin {i+1}/{len(eval_origins)}: "
              f"{origin_date.strftime('%Y-%m-%d')}")

    # Expanding window: all data up to and including origin
    t_idx_all  = prices.index[prices["date"] == origin_date][0]
    train      = prices.iloc[:t_idx_all + 1].copy()
    n_train    = len(train)

    # Current values at origin
    R_t       = train["real_ng"].iloc[-1]    # real TTF price
    spread_t  = train["spread"].iloc[-1]     # log nominal spread
    pi_bar_t  = train["pi_bar"].iloc[-1]     # rolling inflation mean

    for h in HORIZONS:
        y_vals  = []
        sp_vals = []

        for j in range(n_train - h):
            y_j  = train["s_ng"].iloc[j + h] - train["s_ng"].iloc[j]
            sp_j = train["spread"].iloc[j]
            if not np.isnan(y_j) and not np.isnan(sp_j):
                y_vals.append(y_j)
                sp_vals.append(sp_j)

        if len(y_vals) < 3:
            continue

        y_arr  = np.array(y_vals)
        sp_arr = np.array(sp_vals)
        E_pi_h = h * pi_bar_t             # expected cumulative inflation

        # ── Model 40: estimate alpha and beta ─────────────────────────────────
        X40   = np.column_stack([np.ones(len(y_arr)), sp_arr])
        try:
            coef40, _, _, _ = np.linalg.lstsq(X40, y_arr, rcond=None)
            alpha40, beta40 = coef40
            log_chg40 = alpha40 + beta40 * spread_t
            fcst40    = R_t * np.exp(log_chg40 - E_pi_h)
        except Exception:
            fcst40 = np.nan

        # ── Model 41: alpha=0, estimate beta only ─────────────────────────────
        X41   = sp_arr.reshape(-1, 1)
        try:
            coef41, _, _, _ = np.linalg.lstsq(X41, y_arr, rcond=None)
            beta41  = coef41[0]
            log_chg41 = beta41 * spread_t
            fcst41    = R_t * np.exp(log_chg41 - E_pi_h)
        except Exception:
            fcst41 = np.nan

        actual_ym  = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")
        actual_val = get_actual(actual_ym)

        for model_label, fcst in [("PriceSpread(40) α̂,β̂", fcst40),
                                   ("PriceSpread(41) α=0,β̂", fcst41)]:
            records.append({
                "forecast_origin": origin_date.strftime("%Y-%m-%d"),
                "horizon":         h,
                "model":           model_label,
                "actual_month":    actual_ym,
                "forecast":        fcst,
                "actual":          actual_val,
            })

# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("PRICE SPREAD FORECASTING COMPLETE")
print("=" * 60)
print(f"  Total rows:       {len(results)}")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Output:           {OUTPUT_FILE}")
print()

first_origin = results["forecast_origin"].min()
sample = results[results["forecast_origin"] == first_origin]
print(f"Sample — first origin ({first_origin}):")
print(sample[["model","horizon","actual_month",
              "forecast","actual"]].to_string(index=False))

Price Spread forecasting: 133 origins x 2 models x 9 horizons
Models:  Model 40 (alpha+beta), Model 41 (alpha=0, beta only)
Direct h-step regression — separate OLS per horizon
Training: 2006-02-01 onwards  |  pi_bar window: 120 months

  Origin 1/133: 2015-01-31
  Origin 21/133: 2016-09-30
  Origin 41/133: 2018-05-31
  Origin 61/133: 2020-01-31
  Origin 81/133: 2021-09-30
  Origin 101/133: 2023-05-31
  Origin 121/133: 2025-01-31

PRICE SPREAD FORECASTING COMPLETE
  Total rows:       2394
  Forecast origins: 133
  Output:           Output_PriceSpread_forecasts.xlsx

Sample — first origin (2015-01-31):
                 model  horizon actual_month  forecast    actual
 PriceSpread(40) α̂,β̂        1      2015-02 17.620290 22.938516
PriceSpread(41) α=0,β̂        1      2015-02 19.783550 22.938516
 PriceSpread(40) α̂,β̂        3      2015-04 14.018615 22.046423
PriceSpread(41) α=0,β̂        3      2015-04 19.902669 22.046423
 PriceSpread(40) α̂,β̂        6      2015-07 11.063318 20.679393
Pr